# Document Question Answering

An example of using Chroma DB and LangChain to do question answering over documents.

In [16]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.llms import OpenAI
from langchain.chains import VectorDBQA
from langchain.document_loaders import TextLoader
import json
import requests
import re
import json
import torch
import numpy as np
import tqdm as notebook_tqdm
from transformers import AutoTokenizer, AutoModel

## Load documents

Load documents to do question answering over. If you want to do this over your documents, this is the section you should replace.

In [11]:
loader = TextLoader('state_of_the_union.txt')
documents = loader.load()

## Split documents

Split documents into small chunks. This is so we can find the most relevant chunks for a query and pass only those into the LLM.

In [31]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(f"Split into {len(texts)} chunks")
#print(texts[0])

Split into 4 chunks
page_content='# NATURA Zusatzversicherung

NATURA gibt es mit unterschiedlichem Leistungsumfang. Versicherungsschutz, der auf die persÃ¶nliche Situation und auf individuelle AnsprÃ¼che abgestimmt ist.

|                      | **NATURA**                     | **NATURAplus**                |
|----------------------|--------------------------------|--------------------------------|
| **Alternativmedizin**Â¹ |                                |                                |
| FÃ¼r von der CONCORDIA anerkannte Methoden, Heilmittel und Anbieter | Ã„rztliche, naturheilpraktische und therapeutische Behandlungen | Ãœber 70 Methoden | Ãœber 130 Methoden |
| Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel | 75 %, max. CHF 4'000/Jahr | 75 %, max. CHF 6'000/Jahr |
| Ambulante therapeutische Behandlungen inklusive verschriebener Heilmittel | 75 %, max. CHF 1'500/Jahr | 75 %, max. CHF 2'000/Jahr |' metadata={'source': 'state_of_the_union.txt'}


## Initialize ChromaDB

Create embeddings for each chunk and insert into the Chroma vector database.

In [25]:
# ---------------------------
# Step 1: Load Processed Data
# ---------------------------
def load_chunks(json_path):
    """
    Load processed text chunks from a JSON file.
    Expected JSON format: a list of dictionaries with at least a "text" key.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Extract texts; adjust if your JSON structure differs
    texts = [chunk["text"] for chunk in data]
    return texts, data  # also returning the full data for metadata if needed

# ----------------------------------------
# Step 2: Generate Embeddings with Contriever
# ----------------------------------------
def compute_embeddings(texts, tokenizer, model, device="gpu", batch_size=16):
    """
    Compute embeddings for a list of texts using Contriever.
    """
    model.to(device)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            # Tokenize the batch. Adjust max_length if needed.
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            # The Contriever model returns the embeddings in last_hidden_state[:,0,:] or use a pooling strategy.
            # Here, we assume a pooling strategy is implemented (or use mean pooling).
            # For demonstration, we use mean pooling on the token embeddings (excluding padding tokens)
            token_embeddings = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)
            attention_mask = inputs["attention_mask"].unsqueeze(-1)  # shape: (batch_size, seq_len, 1)
            # Avoid division by zero; sum and average over non-padding tokens
            sum_embeddings = torch.sum(token_embeddings * attention_mask, dim=1)
            valid_tokens = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            batch_embeddings = sum_embeddings / valid_tokens
            embeddings.append(batch_embeddings.cpu().numpy())
    embeddings = np.vstack(embeddings)
    return embeddings

if __name__ == "__main__":
    model_name = "facebook/contriever"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Compute embeddings for the texts
    #device = "cuda" if torch.cuda.is_available() else "cpu"
    device = "cpu"
    embeddings = compute_embeddings(texts, tokenizer, model, device=device)
    print(f"Computed embeddings with shape: {embeddings.shape}")

ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [37]:
from langchain_community.embeddings import FakeEmbeddings

In [33]:
embeddings = FakeEmbeddings(size=1352)
vectordb = Chroma.from_documents(texts, embeddings)

## Create the chain

Initialize the chain we will use for question answering.

In [40]:
from langchain_google_genai import ChatGoogleGenerativeAI
# Initialize the HuggingFaceHub LLM
from langchain_google_genai import ChatGoogleGenerativeAI

import os
from getpass import getpass

# Set your API key
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass('Enter your Google API Key: ')

# Initialize the model
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash-exp', temperature=0.7)

# Create the VectorDBQA chain
qa = VectorDBQA.from_chain_type(llm=llm, chain_type="stuff", vectorstore=vectordb)


C:\Users\trist\PycharmProjects\RAG-Studienarbeit\.venv\lib\site-packages\langchain\chains\retrieval_qa\base.py:316: UserWarning: `VectorDBQA` is deprecated - please use `from langchain.chains import RetrievalQA`
  warnings.warn(


## Ask questions!

Now we can use the chain to ask questions!

In [43]:
query = "Was is der Unterschied zwischen Natura und Natura plus?"
qa.run(query)

"Die Zusatzversicherung NATURA und NATURAplus der CONCORDIA unterscheiden sich im Leistungsumfang. Hier eine Zusammenfassung der Unterschiede:\n\n**Alternativmedizin:**\n\n*   **NATURA:**\n    *   Erstattet 75% der Kosten für ärztliche, naturheilpraktische und therapeutische Behandlungen bis zu einem Maximum von CHF 4'000 pro Jahr.\n    *   Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 4'000/Jahr\n    *   Ambulante therapeutische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 1'500/Jahr\n    *   FÃ¼r von der CONCORDIA anerkannte Methoden, Heilmittel und Anbieter: Ãœber 70 Methoden\n*   **NATURAplus:**\n    *   Erstattet 75% der Kosten für ärztliche, naturheilpraktische und therapeutische Behandlungen bis zu einem Maximum von CHF 6'000 pro Jahr.\n    *   Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 6'000/Jahr\n    *   Ambulante therapeutische Behandlungen inklusive versch

In [44]:
print("Die Zusatzversicherung NATURA und NATURAplus der CONCORDIA unterscheiden sich im Leistungsumfang. Hier eine Zusammenfassung der Unterschiede:\n\n**Alternativmedizin:**\n\n*   **NATURA:**\n    *   Erstattet 75% der Kosten für ärztliche, naturheilpraktische und therapeutische Behandlungen bis zu einem Maximum von CHF 4'000 pro Jahr.\n    *   Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 4'000/Jahr\n    *   Ambulante therapeutische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 1'500/Jahr\n    *   FÃ¼r von der CONCORDIA anerkannte Methoden, Heilmittel und Anbieter: Ãœber 70 Methoden\n*   **NATURAplus:**\n    *   Erstattet 75% der Kosten für ärztliche, naturheilpraktische und therapeutische Behandlungen bis zu einem Maximum von CHF 6'000 pro Jahr.\n    *   Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 6'000/Jahr\n    *   Ambulante therapeutische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 2'000/Jahr\n    *   FÃ¼r von der CONCORDIA anerkannte Methoden, Heilmittel und Anbieter: Ãœber 130 Methoden\n\n**Prävention:**\n\n*   **NATURA & NATURAplus:**\n    *   Beide Varianten leisten Beiträge von 90%, maximal CHF 500 pro Jahr, an diverse Präventionsmassnahmen.\n\n**Gesundheitsförderung:**\n\n*   **NATURA:**\n    *   Muskelaufbau (auf ärztliche Verordnung): 50 %, max. CHF 200/Bereich und Jahr\n    *   Kurse zu weiteren Gesundheitsthemen: Mehrere Bereiche: total max. CHF 500/Jahr\n*   **NATURAplus:**\n    *   Fitness (CONCORDIA Sport-Bonus bei NATURAplus, Beiträge an Fitnessabo und MuKi-/VaKi-Turnen)\n    *   Schwangerschaft (z. B. Geburtsvorbereitung)\n    *   Babyschwimmen (bei NATURAplus)\n\nZusammenfassend bietet NATURAplus einen erweiterten Leistungsumfang im Bereich der Alternativmedizin und Gesundheitsförderung im Vergleich zu NATURA.")

Die Zusatzversicherung NATURA und NATURAplus der CONCORDIA unterscheiden sich im Leistungsumfang. Hier eine Zusammenfassung der Unterschiede:

**Alternativmedizin:**

*   **NATURA:**
    *   Erstattet 75% der Kosten für ärztliche, naturheilpraktische und therapeutische Behandlungen bis zu einem Maximum von CHF 4'000 pro Jahr.
    *   Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 4'000/Jahr
    *   Ambulante therapeutische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 1'500/Jahr
    *   FÃ¼r von der CONCORDIA anerkannte Methoden, Heilmittel und Anbieter: Ãœber 70 Methoden
*   **NATURAplus:**
    *   Erstattet 75% der Kosten für ärztliche, naturheilpraktische und therapeutische Behandlungen bis zu einem Maximum von CHF 6'000 pro Jahr.
    *   Ambulante naturheilpraktische Behandlungen inklusive verschriebener Heilmittel: 75 %, max. CHF 6'000/Jahr
    *   Ambulante therapeutische Behandlungen inklusive verschriebener Heil